# 12. Define the observation population and see the actual movement

Which real source videos can answer the new question, and what information does each measured window contain? We will connect the fresh annotations, local video, extracted pose, eligibility rule and source split before inspecting any detection score.

## Run this notebook independently

From the repository root, install the shared environment and start Jupyter:

```bash
uv sync --project experiments/aaai
AAAI_ITERATION02_RUN_DIR=/absolute/path/to/experiments/aaai/artifacts/iteration02/PROTOCOL_HASH \
  uv run --project experiments/aaai jupyter lab experiments/aaai
```

Replace `PROTOCOL_HASH` with the exact completed run you intend to inspect. The setup displays its producer command and verifies its recorded identity before showing results. It never chooses the newest run automatically. Alternatively, put `AAAI_ITERATION02_RUN_DIR` in the repository root `.env`. Do not create another environment file. Python-dotenv calls the real functions `load_dotenv` and `find_dotenv`; the aliases below implement the requested `load_env(find_env())` spelling without printing credentials.

Python 3.12 and the notebook, plotting, data and model dependencies are declared in the existing `pyproject.toml`. Select its kernel. This notebook does not depend on cells from another notebook. A full experiment fits models through the recorded producer command; this tutorial loads those real fitted outputs, checks them and teaches how to interpret them. Missing required outputs stop execution instead of triggering synthetic examples.

A small development run has its own protocol identity. To inspect it deliberately, also set `AAAI_ITERATION02_ALLOW_DEVELOPMENT=1`. Its outputs remain labeled incomplete. Do not use that option to bypass a failed evidence verifier.

The experiment is exploratory because the available sources informed earlier work. A dataset folder label is not a diagnosis independently established here. The focal endpoint is normal versus Parkinsons, stroke, cerebral palsy and myopathic labels. The mandatory broad endpoint uses all non-normal folders. The word “normal” refers to the dataset annotation, not a clinical clearance.

In [ ]:
from pathlib import Path
import os, sys, json
from dotenv import load_dotenv as load_env, find_dotenv as find_env

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ambient").is_dir() and (p / "data/gavd").is_dir())
os.chdir(ROOT)
load_env(find_env())
BASE = ROOT / "experiments/aaai"
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from aaai_gait.iteration02_notebook_support import (
    EvidenceRun, package_versions, show_diagram, show_video, show_status,
    optional_evidence, joint_table, source_role_table, plot_counts,
    prediction_example, example_losses, plot_prediction, plot_adequacy,
    plot_timing, plot_source_scores, within_fold_metrics, plot_metric_table,
    fold_json_records, fold_tables, cohort_table, mask_bank_summary,
    measured_window, plot_paired_differences, timing_video_example, external_replication, plot_condition_confusions,
)

run = EvidenceRun.from_env()
display(run.describe().T.rename(columns={0: "value"}))
display(package_versions())
if run.mode not in {"full", "full_study"}:
    display(Markdown("**DEVELOPMENT RUN. This is incomplete evidence and cannot supply final manuscript results.**"))

## 1. Start with the recorded cohort, then compare it with today's annotations

The experiment reads the current condition CSVs in `data/gavd`. It does not count the `source` distribution cache as another participant. A video may supply several annotated sequences, and each sequence may supply overlapping windows. These are repeated observations of a source, not additional independent people.

The saved manifest fixes the annotations used by this run. We also read the current CSVs and compare sequence, label and annotation hashes. If the dataset has changed, stop and create a new run identity. Do not combine an old score table with a refreshed denominator. Existing poses can be reused only when their extraction provenance matches the annotation and media inputs.

In [ ]:
from aaai_gait.study_data import build_study_manifest
saved_manifest = run.table("manifest.csv", columns=("sequence_id", "source_id", "label", "csv_sha256"))
fresh_manifest = build_study_manifest(ROOT, BASE / "artifacts/poses")
keys = ["sequence_id", "source_id", "label", "csv_sha256"]
saved_keys = saved_manifest[keys].sort_values("sequence_id").reset_index(drop=True)
fresh_keys = fresh_manifest[keys].sort_values("sequence_id").reset_index(drop=True)
assert saved_keys.equals(fresh_keys), "Annotations differ from the verified run. Create a new protocol identity."
display(saved_manifest.groupby("label").agg(sequences=("sequence_id", "nunique"), source_memberships=("source_id", "nunique")))
print("Current annotations match the saved run snapshot.")

## 2. Trace one normal sequence back to its source video

The preview below uses a deterministic source and start time from this run. The overlay uses the stored raw image coordinates, the extraction's frame-number base and its actual video frame rate. It draws only confident observed joints. It does not infer a missing ankle from the picture to improve the result.

Compare the person, crop and skeleton. A pose can look numerically plausible while following the wrong person. Check turning, occlusion, walking aids and camera movement. A failed decode remains a visible availability problem. Previews stay under `data/videos-gavd/<source_id>/previews/` and keep the source playback rate.

In [ ]:
normal_example = show_video(run, label="normal")
display(pd.DataFrame([normal_example]))

## 3. Inspect an annotated abnormal example without diagnosing it

Stroke can leave weakness, altered selective control and repeated compensations. A repeatable compensation can be easy to predict. Parkinsonian walking may have small excursion while timing remains regular. Cerebral palsy includes varied developmental motor limitations, and myopathic labels concern muscle-related limitations rather than one brain lesion. These are reasons to preserve movement size, posture and relative information separately.

The [neurological evidence review](docs/iteration02/advice_neuro.md) connects these hypotheses to primary studies and explains their limits. The example is selected by source order, not by the model's most dramatic error. Change the label to explore another recorded family. Visible asymmetry is an observation; it does not identify the affected pathway, medication state or muscle force.

In [ ]:
abnormal_example = show_video(run, label=os.getenv("AAAI_PREVIEW_LABEL", "stroke"))

## 4. Separate raw availability from the final observation population

The final source registry follows the declared support rules. Invalid coordinates stay missing. A target patch needs four valid frames. Source inclusion additionally requires four distinct physical target patches per lower-leg readout joint. Repeated masks and overlapping windows do not turn the same patch into new support.

Longer clips can help estimate cycles but exclude short videos and reduce the normal sample. The primary 32-sample window spans 31 intervals at 15 Hz, about 2.07 seconds. It does not guarantee two complete gait cycles. The 48-frame timing parent has its own support requirements and is not evidence that every primary source supports the timing analysis.

In [ ]:
registry = run.table("source_registry.csv", columns=("source_id", "binary_label"))
windows = run.table("window_metadata.csv", columns=("source_id", "sequence_id", "label"))
assert not registry.source_id.duplicated().any()
registered_windows = windows.loc[windows.source_id.isin(registry.source_id)]
display(registered_windows.groupby("label").agg(sources=("source_id", "nunique"), sequences=("sequence_id", "nunique"), windows=("source_id", "size")))
display(plot_counts(run)); plt.close("all")
display(cohort_table(run, "sequence_ledger.csv"))
display(cohort_table(run, "source_eligibility.csv"))
display(run.json("timing/census/census.json"))

## 5. Keep the masking allowlist separate from the readout

Every training recipe can hide the same 16 approved joints below. The primary score reads the eight lower-leg joints, indices 25 through 32. This readout change is not permission to silently remove shoulders or hips from the masking grid. The shared body reference already contains body measurements, so an own-leg restriction is conditional on that explicitly declared reference.

The numerical order below is deliberate. Left and right joints are distinct, and the duplicate mention of stroke in the original request does not create a second condition class.

In [ ]:
display(joint_table())
assert joint_table()["BLAZEPOSE_33 index"].is_unique

## 6. Freeze fitting, reference and evaluation roles by source

The fitting normals determine model parameters and development choices. Separate reserved normals estimate the relational reference after those choices are frozen. Outer-test sources supply the final source scores. Non-test abnormal sources can fit the separately reported supervised classifier; they do not fit the normal-only anomaly model.

The same video cannot cross roles within a fold. Known duplicate recordings must also stay together. Unknown repeated people across different videos remain a limitation. All recovered normal sources were previously explored, so reorganizing their folds does not create independent confirmation.

In [ ]:
roles = source_role_table(registry)
display(roles.groupby(["outer_fold", "role", "binary_label"]).source_id.nunique().rename("sources").reset_index())
display(cohort_table(run, "near_duplicate_candidates.csv"))
print("Retained source IDs:", registry.source_id.nunique(), "retained recording groups:", registry.cluster_id.nunique())
show_diagram("execution_gates")

## Preserve the evidence trail

The run directory is the unit of reproducibility. Keep its input hashes, source roles, fitted models, score references, target predictions, source scores, verification record and failed-comparison ledger together. A successful notebook execution establishes that this tutorial can reproduce its displayed analysis. It does not establish a neurological cause or a clinically valid classifier.

The [iteration02 protocol](docs/iteration02/protocol.md), [results](docs/iteration02/results.md), [limitations](docs/iteration02/limitations.md), and [draft02 readiness review](docs/draft02/README.md) must agree with the same run. The earlier draft01 and notebooks 00–11 remain historical evidence. New figures are written as SVG and PDF masters with PNG previews and an input-hash sidecar under this run's `figures/notebooks` directory.

In [ ]:
record_path = run.record('12_observation_population_and_video_context')
print("Recorded the exact real artifacts read by this notebook:", record_path)
print("This notebook did not replace the experiment producer or alter its source roles.")